# 🎛️ Deep Learning Studio · 4 of 5 — Music: Teach a Network to Listen 🎵

**ISOM 839 · Prescriptive Analytics · Suffolk University · Prof. Hasan Arslan**

A music app receives thousands of new tracks a day. Which instrument is playing? Is it happy or sad? What should play next? You'll teach a network to listen, using one beautiful trick: **turn the sound into a picture, then reuse the picture network from Studio 3.**

### 🧑‍🍳 The recipe (the same five steps in every Studio notebook)
| Step | Plain English |
|---|---|
| **1 · Numbers** | Turn the data into numbers |
| **2 · Shape** | Pick a network shape that fits the data |
| **3 · Train** | Let it practice: guess, measure the miss, nudge the knobs, repeat |
| **4 · Test** | Grade it on data it has never seen |
| **5 · Decide** | Use the prediction to make a business decision |

**Time:** about 20 minutes. 🎧 Headphones or speakers make this one much more fun. A GPU helps a little but isn't needed.

| Sign | Means |
|---|---|
| ▶️ **Run me** | press **Shift + Enter**, no typing needed |
| 🤔 **Bet first** | say your guess out loud *before* you run the next cell |
| ✅ **What you should see** | if your output is close to this, move on (small differences are normal) |
| ✏️ **Your turn** | change one thing and re-run |

---
## Setup

In [ ]:
# @title ▶️ Run me: setup (about 20 seconds)
import os, time, warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import keras
from keras import layers
from IPython.display import display, HTML
keras.utils.set_random_seed(839)

class Progress(keras.callbacks.Callback):
    # prints one short line every few rounds of practice
    def __init__(self, every=5):
        super().__init__(); self.every = every
    def on_epoch_end(self, epoch, logs=None):
        if epoch == 0 or (epoch + 1) % self.every == 0:
            print(f'round {epoch + 1:3d}:  ' + '   '.join(f'{k} {v:.3f}' for k, v in logs.items()))

def card(title, rows, color='#0a66c2'):
    body = ''.join(f"<tr><td style='padding:4px 18px 4px 0;color:#555'>{a}</td><td style='padding:4px 0;font-weight:700'>{b}</td></tr>" for a, b in rows)
    display(HTML(f"<div style='font:15px Arial;border-left:6px solid {color};padding:8px 16px;margin:6px 0;max-width:760px'>"
                 f"<div style='font-weight:700;font-size:16px;margin-bottom:4px'>{title}</div><table>{body}</table></div>"))

print('✅ Ready! Keras', keras.__version__)

**✅ What you should see:** `✅ Ready! Keras 3.…`

---
## Step 1 · Numbers — sound is a long row of numbers

A microphone measures air pressure thousands of times a second. Our clips use **8,000 numbers per second**. Each clip is one second: three notes and then a chord, played by one of **six instruments**.

To keep this notebook fast and free of downloads, a small synthesizer below *makes* the instruments from simple recipes (a flute is almost a pure tone; a guitar note rings and dies…). Real recordings are messier, but the method is exactly the same.

In [ ]:
# @title ▶️ Run me: build a tiny synthesizer and make 3,000 clips
from IPython.display import Audio
SR = 8000                                                   # numbers per second
INSTRUMENTS = ['flute', 'clarinet', 'violin', 'organ', 'guitar', 'piano']

def note(freq, dur, inst, rng):
    t = np.arange(int(SR * dur)) / SR
    start = rng.uniform(0, 2 * np.pi, 10)                   # every note starts at a random point in its wave
    def harmonics(amps, phase=0):                           # a note = a base tone plus quieter, higher copies of it
        return sum(a * np.sin(2 * np.pi * k * freq * t + k * phase + start[k]) for k, a in amps if k * freq < SR / 2)
    fade = lambda attack, release: np.minimum(t / attack, 1) * np.minimum((dur - t) / release, 1)
    if inst == 'flute':       # almost a pure tone, a little breath noise, soft start
        return (harmonics([(1, 1), (2, 0.12)]) + 0.05 * rng.normal(size=t.size)) * fade(0.06, 0.05)
    if inst == 'clarinet':    # only the odd copies: a hollow sound
        return harmonics([(1, 1), (3, 0.45), (5, 0.25), (7, 0.12)]) * fade(0.03, 0.04)
    if inst == 'violin':      # every copy, a wobble in pitch (vibrato), slow start
        wobble = (0.006 * freq / 6) * np.sin(2 * np.pi * 6 * t)
        return harmonics([(k, 1 / k) for k in range(1, 9)], phase=wobble) * fade(0.10, 0.06)
    if inst == 'organ':       # steady, bright, starts instantly
        return harmonics([(1, 1), (2, 0.8), (3, 0.6), (4, 0.5), (6, 0.35), (8, 0.3)]) * fade(0.01, 0.01)
    if inst == 'guitar':      # plucked: rings, then dies fast (high copies first)
        return sum(np.sin(2 * np.pi * k * freq * t + start[k]) * np.exp(-t * (3 + 2.5 * k)) / k for k in range(1, 9) if k * freq < SR / 2) * np.minimum(t / 0.004, 1)
    # piano: struck, rings longer than the guitar
    return sum(np.sin(2 * np.pi * k * freq * t + start[k]) * np.exp(-t * (1.2 + 0.5 * k)) / k ** 1.3 for k in range(1, 9) if k * freq < SR / 2) * np.minimum(t / 0.008, 1)

def make_clip(rng, inst=None, mood=None, root_step=None):
    inst = rng.integers(len(INSTRUMENTS)) if inst is None else inst
    mood = rng.integers(2) if mood is None else mood        # 1 = major (happy), 0 = minor (sad)
    root = 220 * 2 ** ((rng.uniform(0, 12) if root_step is None else root_step) / 12)      # any starting pitch
    steps = [0, 4, 7, 12] if mood == 1 else [0, 3, 7, 12]    # the ONLY difference between happy and sad: one note
    order = rng.permutation(3) if root_step is None else [0, 1, 2]                          # three notes one by one, in any order...
    lengths = rng.uniform(0.17, 0.23, 3)                                                    # ...with a slightly uneven rhythm...
    melody = [note(root * 2 ** (steps[i] / 12), d, INSTRUMENTS[inst], rng) for i, d in zip(order, lengths)]
    chord = sum(note(root * 2 ** (st / 12), 1 - lengths.sum(), INSTRUMENTS[inst], rng) for st in steps)   # ...then all four together
    clip = np.pad(np.concatenate(melody + [chord]), (0, SR))[:SR]                           # exactly one second
    clip = clip / np.abs(clip).max() * rng.uniform(0.5, 0.95) + rng.normal(0, rng.uniform(0.01, 0.12), clip.size)   # loudness and hiss vary
    return clip.astype('float32'), inst, mood

rng = np.random.default_rng(839)
clips, instrument, mood = map(np.array, zip(*[make_clip(rng) for _ in range(3000)]))
print('Clips:', clips.shape, ' -> 3,000 clips, each a row of 8,000 numbers')

for k, name in enumerate(INSTRUMENTS):                      # 🎧 press play on each one
    sample, _, _ = make_clip(np.random.default_rng(k), inst=k, mood=1, root_step=3)
    print(name); display(Audio(sample, rate=SR))

**✅ What you should see (and hear):** `Clips: (3000, 8000)` and six small audio players. **Press play on each.** Same notes, six different instruments. Your ear tells them apart instantly. Can a network?

### First try: feed the raw numbers to the network
Treat each clip as a spreadsheet row of 8,000 numbers (like the first try in Studio 3).

**🤔 Bet first:** with six instruments, pure guessing gets 17%. What will this get?

In [ ]:
# @title ▶️ Run me: the raw-numbers network (about 15 seconds)
train, exam = slice(0, 2400), slice(2400, 3000)             # 2,400 clips to practice, 600 for the exam

raw_net = keras.Sequential([keras.Input(shape=(8000,)), layers.Dense(128, activation='relu'), layers.Dense(6, activation='softmax')])
raw_net.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
raw_net.fit(clips[train], instrument[train], epochs=10, batch_size=32, verbose=0)
raw_accuracy = raw_net.evaluate(clips[exam], instrument[exam], verbose=0)[1]
print(f'Practice accuracy: {raw_net.evaluate(clips[train], instrument[train], verbose=0)[1]:.0%}    Exam accuracy: {raw_accuracy:.0%}')

**✅ What you should see:** practice accuracy near **100%**, exam accuracy around **20%**. That gap is **memorizing** in its purest form: it learned the 2,400 practice clips by heart and understood almost nothing. Raw sound numbers wiggle thousands of times a second; the *meaning* is hidden.

### The trick: turn the sound into a picture
A **spectrogram** shows *which pitches are sounding at each moment*. Think of a **player-piano roll**:

- left → right is **time**
- bottom → top is **pitch** (low notes to high notes)
- brighter = **louder**

In [ ]:
# @title ▶️ Run me: turn every clip into a picture
from scipy.signal import stft

def piano_roll(clip, rows=96):
    f, t, Z = stft(clip, fs=SR, nperseg=1024, noverlap=768)            # how loud is each pitch, in each short slice of time?
    centers = 180 * 2 ** (np.arange(rows) / 24)                         # 96 rows, low to high, spaced like piano keys
    width = centers * (2 ** (1 / 24) - 1) + 8
    keys = np.maximum(0, 1 - np.abs(f[None, :] - centers[:, None]) / width[:, None])
    return np.log1p(30 * (keys @ np.abs(Z))).astype('float32')

pictures = np.stack([piano_roll(c) for c in clips])[..., None]
print('Pictures:', pictures.shape, ' -> each clip is now a 96 x 33 picture')

fig, ax = plt.subplots(1, 6, figsize=(15, 3.6))
for k, a in enumerate(ax):
    sample, _, _ = make_clip(np.random.default_rng(k), inst=k, mood=1, root_step=3)
    a.imshow(piano_roll(sample), origin='lower', aspect='auto', cmap='magma'); a.set_title(INSTRUMENTS[k]); a.set_xticks([]); a.set_yticks([])
ax[0].set_ylabel('pitch (low → high)'); ax[0].set_xlabel('time →'); plt.show()

**✅ What you should see:** `Pictures: (3000, 96, 33, 1)` and six pictures of the same notes (three single notes, then the chord). Now you can **see** what you heard:

- You can see the tune: **three steps up, then a chord** (several lines at once).
- **flute:** just the notes themselves, almost nothing above them (a nearly pure tone)
- **clarinet:** a few widely spaced lines above each note (a hollow sound)
- **violin / organ:** a whole ladder of lines above each note (rich, bright)
- **guitar / piano:** each note is brightest at its start, then **fades** (plucked and struck notes die away)

---
## Step 2 · Shape — the picture network from Studio 3

Sound is now a picture, so we use `Conv2D` magnifying glasses again. One addition near the end: a layer of **32 numbers we name `fingerprint`**. It's the network's own short summary of what it heard. We'll use it in Step 5.

In [ ]:
# @title ▶️ Run me: a recipe for building the listener
def build_listener(answers):
    return keras.Sequential([
        keras.Input(shape=(96, 33, 1)),
        layers.Conv2D(16, 3, activation='relu', padding='same'), layers.MaxPooling2D(),
        layers.Conv2D(32, 3, activation='relu', padding='same'), layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation='relu', padding='same'),
        layers.GlobalAveragePooling2D(),                    # "was this pattern anywhere in the picture?"
        layers.Dense(32, activation='relu', name='fingerprint'),
        layers.Dense(answers, activation='softmax'),
    ])

listener = build_listener(answers=6)
listener.compile(optimizer=keras.optimizers.Adam(0.003), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
print('Knobs:  raw-numbers network', f'{raw_net.count_params():,}', '   listener', f'{listener.count_params():,}')

**✅ What you should see:** the raw-numbers network has over a **million** knobs; the listener about **26,000**. Forty times smaller.

---
## Step 3 · Train

In [ ]:
# @title ▶️ Run me: train the listener (about 30 seconds)
history = listener.fit(pictures[train], instrument[train], epochs=15, batch_size=32, validation_split=0.1, verbose=0, callbacks=[Progress(1)])

**✅ What you should see:** 15 lines. `val_accuracy` climbs quickly and reaches about **0.99** by the last rounds (a small dip along the way is normal).

---
## Step 4 · Test — 600 clips it has never heard

In [ ]:
# @title ▶️ Run me: grade it
guess = listener.predict(pictures[exam], verbose=0).argmax(axis=1)
card('Which instrument? Exam accuracy (guessing = 17%)', [
    ('Raw numbers, 1,000,000+ knobs', f'{raw_accuracy:.0%}'),
    ('Sound as a picture, about 26,000 knobs', f'{(guess == instrument[exam]).mean():.1%}')])
display(pd.crosstab(pd.Series(np.array(INSTRUMENTS)[instrument[exam]], name='really'), pd.Series(np.array(INSTRUMENTS)[guess], name='network said')))

**✅ What you should see:** from about **20%** to about **99%**, with forty times fewer knobs. In the table almost everything sits on the diagonal; any slips are between instruments with a similar tone, such as **clarinet ↔ organ** or **guitar ↔ piano**.

**The lesson of this notebook:** the network shape didn't get smarter. **Step 1 did.** Turning data into the *right* numbers is half the job. (Honest note: our synthetic instruments are cleaner than real recordings, where a result in the 80s or 90s would be a good day.)

---
## Step 5 · Decide — what should play next?

Remember the 32-number `fingerprint` layer? Clips that sound alike get similar fingerprints. That gives the app a recommendation rule:

> **Play next = the clips whose fingerprints are closest to the one playing now.**

This is how "more like this" works in music, shopping and search. The fingerprints are called **embeddings**.

In [ ]:
# @title ▶️ Run me: "more like this"
fingerprinter = keras.Model(listener.inputs, listener.get_layer('fingerprint').output)
prints = fingerprinter.predict(pictures[exam], verbose=0)
prints = prints / np.linalg.norm(prints, axis=1, keepdims=True)
closeness = prints @ prints.T; np.fill_diagonal(closeness, -1)          # how similar is every clip to every other clip?
top3 = np.argsort(-closeness, axis=1)[:, :3]

exam_clips, exam_inst, exam_mood = clips[exam], instrument[exam], mood[exam]
print(f'Across all 600 exam clips: {(exam_inst[top3] == exam_inst[:, None]).mean():.0%} of "play next" picks are the same instrument,')
print(f'                           {(exam_mood[top3] == exam_mood[:, None]).mean():.0%} have the same mood (happy / sad).\n')

now = 7                                                                  # ✏️ try other numbers from 0 to 599
print(f'Now playing: {INSTRUMENTS[exam_inst[now]]}'); display(Audio(exam_clips[now], rate=SR))
for k in top3[now]:
    print(f'Play next:   {INSTRUMENTS[exam_inst[k]]}'); display(Audio(exam_clips[k], rate=SR))

**✅ What you should see:** nearly **100%** of the picks are the same instrument, but only about **50%** share the mood: a coin flip. 🎧 Listen: same instrument, different tunes.

Why? **A fingerprint captures what the network was trained to care about.** We trained it on instruments, so that's all it "hears." If the app's goal is mood playlists, we must train for mood. Same lesson as the bakery: **train for the decision you want to make.**

### ✏️ Your turn: same recipe, new question — happy or sad?
The only difference between our happy (major) and sad (minor) clips is **one note of the chord, shifted by one piano key.** 🎧 Listen first, then run.

**🤔 Bet first:** can the same network shape learn it?

In [ ]:
# @title ▶️ Run me: happy or sad? (about 30 seconds)
happy, _, _ = make_clip(np.random.default_rng(5), inst=5, mood=1, root_step=3)
sad, _, _   = make_clip(np.random.default_rng(5), inst=5, mood=0, root_step=3)
print('happy (major)'); display(Audio(happy, rate=SR)); print('sad (minor)'); display(Audio(sad, rate=SR))

mood_listener = build_listener(answers=2)                   # same shape, 2 answers instead of 6
mood_listener.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
mood_listener.fit(pictures[train], mood[train], epochs=15, batch_size=32, validation_split=0.1, verbose=0, callbacks=[Progress(1)])
print(f'\nHappy or sad? Exam accuracy: {mood_listener.evaluate(pictures[exam], mood[exam], verbose=0)[1]:.0%}  (guessing = 50%)')

**✅ What you should see:** for the first few rounds the accuracy hovers near **0.5** (it has no idea). Then, somewhere in the first 8 rounds, it suddenly **"gets it"** and climbs to nearly **100%**. Watching a network go from clueless to competent in a few rounds is one of the small joys of deep learning.

> **The frontier:** OpenAI's **Whisper** turns speech into text in dozens of languages using this same first step (sound → spectrogram picture) followed by a transformer. Music generators run the idea in reverse: they *produce* the sound from a text description.

### 🧠 Check yourself
Answer in your head first, then click to open.

<details><summary><b>1. The raw-numbers network scored 100% on practice and about 20% on the exam. What happened?</b></summary>

**It memorized** (overfitting). With a million knobs and 2,400 clips it learned each practice clip by heart and understood nothing that carries over to new clips.

</details>

<details><summary><b>2. In the sound picture, what do left-to-right, bottom-to-top and brightness mean?</b></summary>

**Time**, **pitch** (low to high) and **loudness**. It's called a spectrogram.

</details>

<details><summary><b>3. "Play next" matched the instrument almost every time but the mood only half the time. Why?</b></summary>

A fingerprint captures **what the network was trained to care about**. We trained it on instruments, so mood never made it into the fingerprint.

</details>

---
## 🎯 What you just did
| Step | Here |
|---|---|
| **Numbers** | sound → 8,000 numbers a second → **a spectrogram picture** (the step that mattered most) |
| **Shape** | the picture network from Studio 3, plus a 32-number fingerprint layer |
| **Train** | 15 rounds on 2,400 clips |
| **Test** | exam accuracy and a mix-up table on 600 new clips |
| **Decide** | "play next" = nearest fingerprints (embeddings) |

**Next:** Studio 5 — **video**: pictures plus time. 🎬